# 12. Matrix (2D Array) Patterns

A matrix is an array of arrays. Interviews test **index discipline**: rotations, spirals, diagonals, in-place changes.

**Learning order:** BASICS → TRANSPOSE AND ROTATE → SPIRAL → SET MATRIX ZEROES → DIAGONAL TRAVERSAL → SEARCH A SORTED MATRIX → INTERVIEW THINKING

## 1. Basics

- `rows = len(m)`, `cols = len(m[0])`. Check for an empty matrix.
- Cell `(r, c)`; neighbours with `dirs = [(1,0),(-1,0),(0,1),(0,-1)]`.
- Row `r`: `m[r]`. Column `c`: `[row[c] for row in m]`.
- Anti-diagonals share `r + c`; main diagonals share `r - c`.
- Transpose: `[list(col) for col in zip(*m)]`.

In [1]:
m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print([list(col) for col in zip(*m)])   # [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
print([row[1] for row in m], [m[i][i] for i in range(3)])   # [2, 5, 8] [1, 5, 9]

[[1, 4, 7], [2, 5, 8], [3, 6, 9]]
[2, 5, 8] [1, 5, 9]


## 2. Rotate Image 90° clockwise, in place

**Transpose** (swap `m[i][j]` with `m[j][i]`), then **reverse each row**. O(n²) time, O(1) space. (Counter-clockwise: reverse each row first, then transpose, or transpose then reverse the rows' order.)

In [2]:
def rotate_image(matrix):
    n = len(matrix)
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
    for row in matrix:
        row.reverse()
    return matrix

print(rotate_image([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))   # [[7, 4, 1], [8, 5, 2], [9, 6, 3]]

[[7, 4, 1], [8, 5, 2], [9, 6, 3]]


## 3. Spiral Order

Keep four boundaries `top, bottom, left, right`. Walk right along the top row, down the right column, left along the bottom row, up the left column, shrinking the boundaries after each side. **Check the boundaries again** before the 3rd and 4th side (for single row/column leftovers).

In [3]:
def spiral_order(matrix):
    res = []
    top, bottom, left, right = 0, len(matrix) - 1, 0, len(matrix[0]) - 1
    while top <= bottom and left <= right:
        for c in range(left, right + 1):
            res.append(matrix[top][c])
        top += 1
        for r in range(top, bottom + 1):
            res.append(matrix[r][right])
        right -= 1
        if top <= bottom:
            for c in range(right, left - 1, -1):
                res.append(matrix[bottom][c])
            bottom -= 1
        if left <= right:
            for r in range(bottom, top - 1, -1):
                res.append(matrix[r][left])
            left += 1
    return res

print(spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))                      # [1, 2, 3, 6, 9, 8, 7, 4, 5]
print(spiral_order([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]))          # [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]

[1, 2, 3, 6, 9, 8, 7, 4, 5]
[1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]


## 4. Set Matrix Zeroes (O(1) extra space)

If a cell is `0`, zero its whole row and column. Marking in a new matrix costs O(m·n). Trick: use the **first row and first column as markers**.
1. Remember whether the first row / first column originally contain a zero.
2. For every other cell that is `0`, set `row[0]` and `col[0]` marks to `0`.
3. Zero cells according to the marks.
4. Finally zero the first row/column if needed.

In [4]:
def set_zeroes(matrix):
    rows, cols = len(matrix), len(matrix[0])
    first_row_zero = any(matrix[0][c] == 0 for c in range(cols))
    first_col_zero = any(matrix[r][0] == 0 for r in range(rows))
    for r in range(1, rows):
        for c in range(1, cols):
            if matrix[r][c] == 0:
                matrix[r][0] = matrix[0][c] = 0
    for r in range(1, rows):
        for c in range(1, cols):
            if matrix[r][0] == 0 or matrix[0][c] == 0:
                matrix[r][c] = 0
    if first_row_zero:
        for c in range(cols):
            matrix[0][c] = 0
    if first_col_zero:
        for r in range(rows):
            matrix[r][0] = 0
    return matrix

print(set_zeroes([[1, 1, 1], [1, 0, 1], [1, 1, 1]]))   # [[1, 0, 1], [0, 0, 0], [1, 0, 1]]

[[1, 0, 1], [0, 0, 0], [1, 0, 1]]


## 5. Diagonal Traverse

Elements on the same anti-diagonal share `r + c`. Collect each diagonal, **reverse every other one** (zigzag).

In [5]:
def diagonal_traverse(matrix):
    rows, cols = len(matrix), len(matrix[0])
    res = []
    for d in range(rows + cols - 1):
        diag = [matrix[r][d - r] for r in range(max(0, d - cols + 1), min(rows, d + 1))]
        res.extend(diag[::-1] if d % 2 == 0 else diag)
    return res

print(diagonal_traverse([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))   # [1, 2, 4, 7, 5, 3, 6, 8, 9]

[1, 2, 4, 7, 5, 3, 6, 8, 9]


## 6. Search a 2D Matrix II (rows and columns sorted)

**Staircase search** from the **top-right** corner: if the value is bigger than the target, move **left** (the whole column below is even bigger); if smaller, move **down** (the whole row to the left is even smaller). O(m + n).

In [6]:
def search_matrix_ii(matrix, target):
    r, c = 0, len(matrix[0]) - 1
    while r < len(matrix) and c >= 0:
        if matrix[r][c] == target:
            return True
        if matrix[r][c] > target:
            c -= 1
        else:
            r += 1
    return False

grid = [[1, 4, 7, 11, 15], [2, 5, 8, 12, 19], [3, 6, 9, 16, 22], [10, 13, 14, 17, 24], [18, 21, 23, 26, 30]]
print(search_matrix_ii(grid, 5), search_matrix_ii(grid, 20))   # True False

True False


## 7. Interview thinking

**Signals:** "rotate", "spiral", "diagonal", "zero out rows/columns", "sorted rows and columns".

**Say:** "I'll use four boundaries and shrink them" / "I'll reuse the first row and column as markers to stay O(1) space" / "I start at the corner where one move decreases and the other increases."

**Common mistakes:** `[[0]*n]*m` (aliasing), off-by-one on boundaries, not rechecking boundaries inside the spiral loop, assuming square matrices, using extra space when O(1) is required, mixing `row`/`col` order (`matrix[r][c]`).

## Quick Revision

- Rotate 90° clockwise = transpose + reverse each row.
- Spiral = 4 shrinking boundaries, recheck before bottom/left.
- Set zeroes O(1) = first row/column as markers.
- Diagonals share `r + c`. Sorted-both-ways search: staircase from top-right.